# 📈 Sales & Demand Forecasting for Businesses

A comprehensive sales forecasting system using historical retail data.

**What this notebook covers:**
1. **Synthetic Dataset Generation** — 3 years of realistic daily retail sales
2. **Data Cleaning** — Handling missing values, outliers, and data quality
3. **Exploratory Data Analysis** — Trends, seasonality, and patterns
4. **Seasonal Decomposition** — Breaking down time-series components
5. **Feature Engineering** — Calendar, lag, and rolling features
6. **ML Models** — Linear Regression, Random Forest, Gradient Boosting
7. **SARIMA Model** — Classical time-series forecasting
8. **Model Comparison** — MAE, RMSE, MAPE, R² evaluation
9. **Forecast Dashboard** — Business-friendly visual output
10. **Feature Importance** — What drives sales predictions
11. **Business Insights** — Revenue summaries & actionable recommendations

---
## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

# --- Plotting style ---
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
})

# Color palette
COLORS = {
    'primary': '#6366f1',
    'secondary': '#10b981',
    'accent': '#f59e0b',
    'danger': '#ef4444',
    'info': '#3b82f6',
    'purple': '#8b5cf6',
    'pink': '#ec4899',
}

np.random.seed(42)
print('✅ All libraries loaded successfully!')

---
## 2. Synthetic Sales Dataset Generation

We generate **3 years of daily retail sales data** (Jan 2022 – Dec 2024) with realistic patterns:
- **Trend**: Gradual upward growth over time
- **Seasonality**: Monthly & weekly cycles (weekends higher, holiday spikes)
- **Noise**: Random daily fluctuations
- **Product categories**: Electronics, Clothing, Groceries

In [ ]:
# --- Generate date range ---
dates = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')
n = len(dates)

# --- Base sales with upward trend ---
trend = np.linspace(500, 800, n)  # gradual growth from $500 to $800/day

# --- Monthly seasonality (higher in Nov-Dec for holidays, lower in Jan-Feb) ---
month_effect = 80 * np.sin(2 * np.pi * np.arange(n) / 365.25 - np.pi / 2)

# --- Weekly seasonality (weekends are ~20% higher) ---
day_of_week = dates.dayofweek
weekly_effect = np.where(day_of_week >= 5, 60, -15)  # Sat/Sun boost

# --- Holiday spikes (Black Friday, Christmas, New Year) ---
holiday_spike = np.zeros(n)
for i, d in enumerate(dates):
    if d.month == 11 and d.day >= 24 and d.day <= 30:  # Black Friday week
        holiday_spike[i] = 200
    elif d.month == 12 and d.day >= 20:  # Christmas rush
        holiday_spike[i] = 250
    elif d.month == 1 and d.day <= 5:  # New Year sales
        holiday_spike[i] = 100

# --- Random noise ---
noise = np.random.normal(0, 40, n)

# --- Combine all components ---
sales = trend + month_effect + weekly_effect + holiday_spike + noise
sales = np.maximum(sales, 50)  # floor at $50

# --- Create DataFrame ---
df = pd.DataFrame({
    'date': dates,
    'sales': np.round(sales, 2),
})
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# --- Add product category breakdown ---
df['electronics'] = np.round(df['sales'] * np.random.uniform(0.35, 0.45, n), 2)
df['clothing'] = np.round(df['sales'] * np.random.uniform(0.25, 0.35, n), 2)
df['groceries'] = np.round(df['sales'] - df['electronics'] - df['clothing'], 2)

# --- Inject ~2% missing values for realism ---
mask = np.random.random(n) < 0.02
df.loc[mask, 'sales'] = np.nan

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'Missing values: {df["sales"].isna().sum()} ({df["sales"].isna().mean():.1%})')
print(f'\nFirst 5 rows:')
df.head()

---
## 3. Data Cleaning

Handle missing values using forward-fill and detect/cap outliers with IQR method.

In [ ]:
# --- Fill missing values ---
missing_before = df['sales'].isna().sum()
df['sales'] = df['sales'].ffill().bfill()
print(f'Missing values filled: {missing_before} → {df["sales"].isna().sum()}')

# --- Outlier detection and capping (IQR method) ---
Q1 = df['sales'].quantile(0.25)
Q3 = df['sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ((df['sales'] < lower_bound) | (df['sales'] > upper_bound)).sum()
df['sales'] = df['sales'].clip(lower=lower_bound, upper=upper_bound)

print(f'Outliers capped: {outliers}')
print(f'Clean data range: ${df["sales"].min():.0f} – ${df["sales"].max():.0f}')
print(f'\nSales statistics after cleaning:')
df['sales'].describe()

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# --- 4a. Overall Sales Trend ---
axes[0, 0].plot(df.index, df['sales'], color=COLORS['primary'], linewidth=0.7, alpha=0.6, label='Daily Sales')
ma_30 = df['sales'].rolling(30).mean()
axes[0, 0].plot(df.index, ma_30, color=COLORS['danger'], linewidth=2.5, label='30-Day Moving Avg')
axes[0, 0].set_title('📈 Overall Sales Trend', fontsize=14)
axes[0, 0].set_ylabel('Sales ($)')
axes[0, 0].legend(loc='upper left')

# --- 4b. Monthly Sales Distribution ---
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_monthly = df.copy()
df_monthly['month'] = df_monthly.index.month
monthly_colors = [COLORS['primary'] if m not in [11,12] else COLORS['accent'] for m in range(1,13)]
bp = axes[0, 1].boxplot([df_monthly[df_monthly['month']==m]['sales'] for m in range(1,13)],
                        labels=month_names, patch_artist=True, widths=0.6)
for patch, color in zip(bp['boxes'], monthly_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0, 1].set_title('📊 Monthly Sales Distribution', fontsize=14)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Sales ($)')

# --- 4c. Day of Week Pattern ---
df_dow = df.copy()
df_dow['day_of_week'] = df_dow.index.dayofweek
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_avg = df_dow.groupby('day_of_week')['sales'].mean()
colors_dow = [COLORS['primary']] * 5 + [COLORS['secondary']] * 2
bars = axes[1, 0].bar(dow_names, dow_avg.values, color=colors_dow, edgecolor='white', linewidth=2)
for bar, val in zip(bars, dow_avg.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 3,
                    f'${val:.0f}', ha='center', fontsize=9, fontweight='bold')
axes[1, 0].set_title('📅 Average Sales by Day of Week', fontsize=14)
axes[1, 0].set_ylabel('Avg Sales ($)')

# --- 4d. Category Trends ---
for cat, color in zip(['electronics','clothing','groceries'],
                       [COLORS['primary'], COLORS['pink'], COLORS['secondary']]):
    ma = df[cat].rolling(30).mean()
    axes[1, 1].plot(df.index, ma, linewidth=2, label=cat.title(), color=color)
axes[1, 1].set_title('🛍️ Category Trends (30-Day MA)', fontsize=14)
axes[1, 1].set_ylabel('Sales ($)')
axes[1, 1].legend()

plt.suptitle('🔍 Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/eda_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ EDA dashboard saved to eda_dashboard.png')

---
## 5. Seasonal Decomposition

Decompose the weekly-resampled sales into **trend**, **seasonal**, and **residual** components.

In [ ]:
# Resample to weekly for cleaner decomposition
weekly_sales = df['sales'].resample('W').mean()

decomposition = seasonal_decompose(weekly_sales, model='additive', period=52)

fig, axes = plt.subplots(4, 1, figsize=(16, 12))

components = [
    ('Observed', decomposition.observed, COLORS['primary']),
    ('Trend', decomposition.trend, COLORS['danger']),
    ('Seasonal', decomposition.seasonal, COLORS['secondary']),
    ('Residual', decomposition.resid, COLORS['accent']),
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data, color=color, linewidth=1.8)
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.set_ylabel('Sales ($)')
    if title == 'Residual':
        ax.axhline(y=0, color='gray', linewidth=1, linestyle='--')

plt.suptitle('🔄 Seasonal Decomposition (Weekly)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/seasonal_decomposition.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Seasonal decomposition saved to seasonal_decomposition.png')

---
## 6. Time-Based Feature Engineering

Create features that capture temporal patterns:
- **Calendar**: year, month, day of week, quarter, weekend flags
- **Cyclical encoding**: sin/cos transforms of month and day of week
- **Lag features**: 1, 7, 30-day lagged sales
- **Rolling statistics**: 7, 30, 90-day moving average and standard deviation
- **Expanding mean**: cumulative average sales

In [ ]:
df_feat = df[['sales']].copy()

# --- Calendar features ---
df_feat['year'] = df_feat.index.year
df_feat['month'] = df_feat.index.month
df_feat['day_of_week'] = df_feat.index.dayofweek
df_feat['day_of_month'] = df_feat.index.day
df_feat['quarter'] = df_feat.index.quarter
df_feat['is_weekend'] = (df_feat.index.dayofweek >= 5).astype(int)
df_feat['is_month_start'] = df_feat.index.is_month_start.astype(int)
df_feat['is_month_end'] = df_feat.index.is_month_end.astype(int)

# --- Cyclical encoding (sin/cos) ---
df_feat['month_sin'] = np.sin(2 * np.pi * df_feat['month'] / 12)
df_feat['month_cos'] = np.cos(2 * np.pi * df_feat['month'] / 12)
df_feat['dow_sin'] = np.sin(2 * np.pi * df_feat['day_of_week'] / 7)
df_feat['dow_cos'] = np.cos(2 * np.pi * df_feat['day_of_week'] / 7)

# --- Lag features ---
for lag in [1, 7, 14, 30]:
    df_feat[f'lag_{lag}'] = df_feat['sales'].shift(lag)

# --- Rolling statistics ---
for window in [7, 30, 90]:
    df_feat[f'rolling_mean_{window}'] = df_feat['sales'].rolling(window).mean()
    df_feat[f'rolling_std_{window}'] = df_feat['sales'].rolling(window).std()

# --- Expanding mean ---
df_feat['expanding_mean'] = df_feat['sales'].expanding().mean()

# --- Drop rows with NaN from lag/rolling ---
df_feat = df_feat.dropna()

print(f'Feature matrix shape: {df_feat.shape}')
print(f'\nEngineered features ({df_feat.shape[1] - 1} total):')
print([c for c in df_feat.columns if c != 'sales'])

---
## 7. ML Forecasting Models

We train three models with **chronological 80/20 train-test split** (no future data leakage):
1. **Linear Regression** — Simple baseline
2. **Random Forest** — Ensemble learning
3. **XGBoost** — Gradient boosting

In [ ]:
# --- Chronological train-test split (80/20) ---
feature_cols = [c for c in df_feat.columns if c != 'sales']
X = df_feat[feature_cols]
y = df_feat['sales']

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

split_date = X_test.index.min()
print(f'Training: {len(X_train)} days  ({X_train.index.min().date()} to {X_train.index.max().date()})')
print(f'Testing:  {len(X_test)} days  ({X_test.index.min().date()} to {X_test.index.max().date()})')
print(f'Split date: {split_date.date()}')

In [ ]:
# --- Helper: evaluation metrics ---
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    print(f'\n🔹 {name}:')
    print(f'   MAE:  ${mae:,.2f}')
    print(f'   RMSE: ${rmse:,.2f}')
    print(f'   MAPE: {mape:.2f}%')
    print(f'   R²:   {r2:.4f}')
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'preds': y_pred}

results = {}

# --- 1. Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
results['Linear Regression'] = evaluate_model('Linear Regression', y_test, lr_preds)

# --- 2. Random Forest ---
rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5,
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
results['Random Forest'] = evaluate_model('Random Forest', y_test, rf_preds)

# --- 3. XGBoost ---
xgb = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                    subsample=0.8, colsample_bytree=0.8,
                    random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
results['XGBoost'] = evaluate_model('XGBoost', y_test, xgb_preds)

---
## 8. SARIMA Time-Series Model

Fit a **SARIMA(1,1,1)(1,1,1,52)** model on weekly-resampled sales data to capture seasonal weekly patterns.

In [ ]:
# --- Resample to weekly and split ---
weekly = df['sales'].resample('W').mean()
train_size = int(len(weekly) * 0.8)
train_weekly = weekly[:train_size]
test_weekly = weekly[train_size:]

print(f'Weekly training: {len(train_weekly)} weeks')
print(f'Weekly testing:  {len(test_weekly)} weeks')

# --- Fit SARIMA ---
print('\n⏳ Fitting SARIMA model (this may take a minute)...')
sarima = SARIMAX(train_weekly, order=(1, 1, 1), seasonal_order=(1, 1, 1, 52),
                 enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima.fit(disp=False, maxiter=200)
print('✅ SARIMA model fitted!')

# --- Forecast ---
sarima_forecast = sarima_fit.forecast(steps=len(test_weekly))
sarima_result = evaluate_model('SARIMA (Weekly)', test_weekly, sarima_forecast.values)
results['SARIMA'] = sarima_result

# --- Confidence intervals ---
sarima_pred_full = sarima_fit.get_forecast(steps=len(test_weekly))
sarima_ci = sarima_pred_full.conf_int()

---
## 9. Model Comparison & Error Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Only compare ML models (daily) side by side
ml_models = {k: v for k, v in results.items() if k != 'SARIMA'}
model_names = list(ml_models.keys())
colors = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]

# --- 9a. MAE Comparison ---
maes = [ml_models[n]['mae'] for n in model_names]
bars = axes[0].bar(model_names, maes, color=colors, edgecolor='white', linewidth=2)
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f'${val:.2f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('MAE (Lower = Better)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mean Absolute Error ($)')

# --- 9b. MAPE Comparison ---
mapes = [ml_models[n]['mape'] for n in model_names]
bars = axes[1].bar(model_names, mapes, color=colors, edgecolor='white', linewidth=2)
for bar, val in zip(bars, mapes):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f'{val:.2f}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('MAPE (Lower = Better)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Mean Absolute % Error')

# --- 9c. R² Comparison ---
r2s = [ml_models[n]['r2'] for n in model_names]
bars = axes[2].bar(model_names, r2s, color=colors, edgecolor='white', linewidth=2)
for bar, val in zip(bars, r2s):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('R² Score (Higher = Better)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('R² Score')
axes[2].set_ylim(0, 1.1)

plt.suptitle('🏆 ML Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

# --- Summary table ---
print('\n📊 Full Results Summary:')
summary_df = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE ($)': [results[m]['mae'] for m in results],
    'RMSE ($)': [results[m]['rmse'] for m in results],
    'MAPE (%)': [results[m]['mape'] for m in results],
    'R²': [results[m]['r2'] for m in results],
}).set_index('Model')
print(summary_df.round(4).to_string())

---
## 10. Business-Friendly Forecast Visualizations

In [ ]:
# Select best ML model (lowest MAPE)
best_ml = min(ml_models, key=lambda k: ml_models[k]['mape'])
best_preds = ml_models[best_ml]['preds']
print(f'🏆 Best ML model: {best_ml}')

fig, axes = plt.subplots(2, 1, figsize=(18, 12))

# --- 10a. Daily ML Forecast ---
axes[0].plot(y_train.index, y_train, color=COLORS['primary'], linewidth=0.8, alpha=0.5, label='Training Data')
axes[0].plot(y_test.index, y_test, color=COLORS['info'], linewidth=1.2, label='Actual (Test)')
axes[0].plot(y_test.index, best_preds, color=COLORS['danger'], linewidth=1.5,
             linestyle='--', label=f'Forecast ({best_ml})')
axes[0].axvline(x=split_date, color='gray', linewidth=2, linestyle=':', alpha=0.7)
axes[0].text(split_date, axes[0].get_ylim()[1] * 0.95, '  Forecast →',
             fontsize=10, color='gray', fontweight='bold')
axes[0].set_title(f'📈 Daily Sales Forecast — {best_ml}', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Sales ($)', fontweight='bold')
axes[0].legend(loc='upper left', fontsize=10)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# --- 10b. Weekly SARIMA Forecast with Confidence Interval ---
axes[1].plot(train_weekly.index, train_weekly, color=COLORS['primary'],
             linewidth=1.2, alpha=0.5, label='Training Data')
axes[1].plot(test_weekly.index, test_weekly, color=COLORS['info'],
             linewidth=1.5, label='Actual (Test)')
axes[1].plot(test_weekly.index, sarima_forecast.values, color=COLORS['danger'],
             linewidth=2, linestyle='--', label='SARIMA Forecast')
axes[1].fill_between(test_weekly.index,
                     sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1],
                     color=COLORS['danger'], alpha=0.15, label='95% Confidence Interval')
axes[1].set_title('📈 Weekly Sales Forecast — SARIMA', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sales ($)', fontweight='bold')
axes[1].legend(loc='upper left', fontsize=10)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.suptitle('🏢 Business Sales Forecast Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/forecast_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Forecast dashboard saved to forecast_dashboard.png')

---
## 11. Feature Importance Analysis

In [ ]:
# --- Random Forest Feature Importance ---
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
top_features = importance.tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_features.index, top_features.values, color=COLORS['primary'],
               edgecolor='white', linewidth=1.5, height=0.7)

# Highlight top 3
for bar in bars[-3:]:
    bar.set_color(COLORS['accent'])

for bar, val in zip(bars, top_features.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2.,
            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

ax.set_title('🎯 Top 15 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Feature importance chart saved to feature_importance.png')

---
## 12. Monthly Revenue Summary & Business Insights

In [ ]:
# --- Monthly aggregation ---
monthly_rev = df['sales'].resample('ME').agg(['sum', 'mean', 'std', 'count'])
monthly_rev.columns = ['Total Revenue', 'Avg Daily Sales', 'Std Dev', 'Days']

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# --- 12a. Monthly Revenue Bars ---
bar_colors = [COLORS['primary'] if v < monthly_rev['Total Revenue'].quantile(0.75)
              else COLORS['accent'] for v in monthly_rev['Total Revenue']]
axes[0, 0].bar(monthly_rev.index, monthly_rev['Total Revenue'], color=bar_colors,
               edgecolor='white', linewidth=0.5, width=25)
axes[0, 0].set_title('💰 Monthly Revenue', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Revenue ($)')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# --- 12b. Year-over-Year Growth ---
monthly_rev['YoY_Growth'] = monthly_rev['Total Revenue'].pct_change(12) * 100
yoy = monthly_rev['YoY_Growth'].dropna()
yoy_colors = [COLORS['secondary'] if v >= 0 else COLORS['danger'] for v in yoy]
axes[0, 1].bar(yoy.index, yoy.values, color=yoy_colors, edgecolor='white', linewidth=0.5, width=25)
axes[0, 1].axhline(y=0, color='gray', linewidth=1)
axes[0, 1].set_title('📊 Year-over-Year Growth %', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Growth (%)')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# --- 12c. Category Market Share ---
cat_totals = [df['electronics'].sum(), df['clothing'].sum(), df['groceries'].sum()]
cat_labels = ['Electronics', 'Clothing', 'Groceries']
cat_colors = [COLORS['primary'], COLORS['pink'], COLORS['secondary']]
wedges, texts, autotexts = axes[1, 0].pie(cat_totals, labels=cat_labels, colors=cat_colors,
                                           autopct='%1.1f%%', startangle=90,
                                           textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[1, 0].set_title('🎯 Category Market Share', fontsize=13, fontweight='bold')

# --- 12d. Quarterly Revenue Heatmap ---
df_q = df.copy()
df_q['year'] = df_q.index.year
df_q['quarter'] = df_q.index.quarter
q_revenue = df_q.groupby(['year', 'quarter'])['sales'].sum().unstack()
q_revenue.columns = ['Q1', 'Q2', 'Q3', 'Q4']
sns.heatmap(q_revenue, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=2, linecolor='white', ax=axes[1, 1],
            cbar_kws={'label': 'Revenue ($)'})
axes[1, 1].set_title('🗓️ Quarterly Revenue Heatmap', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Year')

plt.suptitle('📋 Business Performance Dashboard', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/business_insights.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Business insights dashboard saved to business_insights.png')

In [ ]:
# --- Print Business Summary ---
total_revenue = df['sales'].sum()
avg_daily = df['sales'].mean()
best_month = monthly_rev['Total Revenue'].idxmax()
worst_month = monthly_rev['Total Revenue'].idxmin()
best_day = df['sales'].idxmax()
weekend_avg = df[df.index.dayofweek >= 5]['sales'].mean()
weekday_avg = df[df.index.dayofweek < 5]['sales'].mean()

print('=' * 70)
print('📊 BUSINESS FORECAST SUMMARY REPORT')
print('=' * 70)
print(f'\n📅 Data Period: {df.index.min().date()} to {df.index.max().date()}')
print(f'📈 Total Revenue: ${total_revenue:,.0f}')
print(f'📊 Average Daily Sales: ${avg_daily:,.0f}')
print(f'\n🏆 Best Month: {best_month.strftime("%B %Y")} (${monthly_rev.loc[best_month, "Total Revenue"]:,.0f})')
print(f'📉 Worst Month: {worst_month.strftime("%B %Y")} (${monthly_rev.loc[worst_month, "Total Revenue"]:,.0f})')
print(f'\n📅 Weekend Avg: ${weekend_avg:,.0f} vs Weekday Avg: ${weekday_avg:,.0f}')
print(f'   Weekend premium: +{((weekend_avg/weekday_avg)-1)*100:.1f}%')

print(f'\n🏆 Best Forecasting Model: {best_ml}')
print(f'   MAPE: {results[best_ml]["mape"]:.2f}%')
print(f'   R²:   {results[best_ml]["r2"]:.4f}')

print('\n' + '=' * 70)
print('💡 KEY BUSINESS RECOMMENDATIONS')
print('=' * 70)
print('\n1. 📦 INVENTORY: Stock ~15-20% more during Nov-Dec for holiday demand')
print('2. 👥 STAFFING: Schedule extra staff on weekends (higher sales volume)')
print('3. 💰 PRICING: Consider dynamic pricing during peak seasons')
print('4. 📊 FORECASTING: Use the best model for 30-60 day rolling forecasts')
print('5. 🎯 CATEGORIES: Electronics leads revenue — prioritize marketing')
print('\n' + '=' * 70)
print('✅ Forecast report complete!')